# GCP Pub/Sub Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/gcp_pubsub/pubsub_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/gcp_pubsub/pubsub_demo.ipynb)

## Business Scenario

Pub/Sub is the backbone of GCP eventing. You need validated ingestion for telemetry and event data.

## Value Proposition

- Validated event ingestion for Pub/Sub
- Contract-based schema enforcement
- Consistent outputs for analytics

---

## Goals

1. Subscribe to a Pub/Sub topic
2. Validate events
3. Materialize clean records


## 🚀 Step 1: Setup GCP Pub/Sub

Before running this notebook, you need:
1. A GCP Pub/Sub topic and subscription
2. Application Default Credentials configured (via `gcloud auth application-default login`)
3. Your GCP project ID

Set your environment variable:
```bash
export GCP_PROJECT_ID="my-gcp-project"
```

## 📝 Step 2: Review the Contract

Our contract defines the expected IoT telemetry schema and quality rules.

In [ ]:
with open('pubsub_contract.yaml', 'r') as f:
    print("📄 Pub/Sub Contract:")
    print("-------------------")
    print(f.read())

## ▶️ Step 3: Start the Pub/Sub Subscriber

This will connect to your Pub/Sub subscription and start processing messages.

In [ ]:
from lakelogic.core.streaming_processor import StreamingDataProcessor
import threading
import time

# Initialize processor
processor = StreamingDataProcessor(
    contract="pubsub_contract.yaml",
    framework="bytewax"
)

# Start in background
thread = threading.Thread(target=processor.start)
thread.daemon = True
thread.start()

print("🚀 Pub/Sub subscriber started!")
print("Waiting for IoT telemetry...")
time.sleep(5)

## 🧪 Step 4: Publish Test Messages

Let's publish some test IoT telemetry to the topic.

In [ ]:
from google.cloud import pubsub_v1
import json
import os
from datetime import datetime

project_id = os.getenv("GCP_PROJECT_ID")
topic_id = "iot-telemetry-topic"  # Update with your topic ID

if project_id:
    publisher = pubsub_v1.PublisherClient()
    topic_path = publisher.topic_path(project_id, topic_id)
    
    test_readings = [
        {
            "deviceId": "SENSOR-001",
            "temperature": 22.5,
            "humidity": 45.2,
            "pressure": 1013.25,
            "timestamp": datetime.now().isoformat(),
            "location": {"lat": 37.7749, "lon": -122.4194}
        },
        {
            "deviceId": "SENSOR-002",
            "temperature": 24.1,
            "humidity": 52.8,
            "timestamp": datetime.now().isoformat()
        }
    ]
    
    for reading in test_readings:
        data = json.dumps(reading).encode("utf-8")
        future = publisher.publish(topic_path, data)
        print(f"📤 Published reading from {reading['deviceId']} - MessageId: {future.result()}")
    
    print("\n✅ Test messages published! Check LakeLogic logs...")
else:
    print("ℹ️  Set GCP_PROJECT_ID to publish test messages")

## 📊 Step 5: Verify Results

Check the materialized Delta table.

In [ ]:
import polars as pl
import time

# Wait for processing
time.sleep(3)

try:
    df = pl.read_delta("./data/bronze/gcp_iot/")
    print("📂 Processed Telemetry:")
    print(df)
except Exception as e:
    print(f"ℹ️  No data yet: {e}")

## 🎉 Summary

You just:
- ✅ Connected to GCP Pub/Sub
- ✅ Validated IoT telemetry against a contract
- ✅ Materialized events to Delta Lake

This pattern enables **real-time IoT analytics** at global scale!